# GVH–CHH — Jupiter Horizons H0: Force Ablation and Time-Window Robustness Gate (FAST)

**Notebook canonique :** `GVH_CHH_Jupiter_Horizons_H0_Force_Ablation_And_Time_Window_Robustness_Gate_FAST`  
**Version :** v0.2 (2026-11-20), correctif technique de la phase finale (sans changement des équations, forces, fenêtres ni paramètres expérimentaux). **Ce fichier n’est pas pré-validé : il doit être exécuté avec les CSV JPL originaux pour établir un verdict.**  
**Prédécesseur :** `GVH_CHH_Jupiter_Horizons_Full_Force_Model_And_Residual_Calibration_Gate_FAST(3).ipynb`, sortie JPL : Kepler 224,670299 km, J2+interactions 12,859714 km, J2/J4/J6+interactions+Soleil 12,946623 km (RMSE de coordonnée sur données réservées).

**Objectif :** ablation contrôlée des forces connues et robustesse de la prédiction sur plusieurs fenêtres d'entraînement et sous-intervalles réservés. **Aucun objet caché n'est autorisé**. L'expérience compare une intégration newtonienne simplifiée à des **éphémérides de référence calculées par JPL**, pas à des mesures astrométriques brutes indépendantes. Les coefficients gravitationnels sont fixés et approximatifs. Ce n'est ni une EOM-GVH ni une validation de la nouveauté CHH.

**Conserver l'architecture à quatre niveaux :** (1) CHH / problème de reconstruction ; (2) forces physiques établies ; (3) test numérique, gardes, résidus ; (4) verdict avec verrous ouverts.

**Dépendance obligatoire :** récupérer *les cinq fichiers CSV JPL et le JSON source* créés par le notebook précédent. Ne pas réutiliser une copie du `.ipynb` en croyant qu'elle contient ces fichiers. Aucun remplacement synthétique silencieux n'est permis. Le modèle et les fenêtres sont sélectionnés **avant** d'inspecter les résultats de la présente ablation.

## Niveau 1 — Plan d'expérience préfixé

- Plan de base : **deux fenêtres d'entraînement** de 10 et 15 jours, avec réserve commune de jours **strictement supérieurs à 18**. 10 j et 15 j permettent de tester la dépendance à la quantité d'entraînement. La réserve commune garantit une comparaison des prévisions sur les mêmes dates.
- En option étendue, une troisième fenêtre (12 j) et deux variantes supplémentaires sont ajoutées **avant exécution**.
- Cibles évaluées : Io et Ganymède ; Europe et Callisto sont des perturbateurs connus, leur état initial est fixe. **12 degrés de liberté** par modèle (positions/vitesses initiales des deux cibles uniquement).
- Ablations de base : `KEPLER`, `J2_ONLY`, `MUTUAL_ONLY`, `J2_MUTUAL`, `J2_MUTUAL_SUN`, `J2_J4_J6_MUTUAL_SUN`. Les effets du Soleil et des termes J4/J6 sont séparés conditionnellement aux forces déjà incluses.
- Toutes les variantes utilisent exactement les mêmes CSV, unités, paramètres prédéfinis, budget d'ajustement et frontières temporelles. Le Soleil est une **force externe prescrite** provenant des éphémérides JPL sur toute la fenêtre ; ce n'est pas une prédiction autonome du Soleil.
- Réserves diagnostiques supplémentaires : jour +1 à +3 après la frontière d'entraînement (lorsque disponibles) ; **non indépendantes** de la réserve commune, sans optimisation a posteriori.
- **Ne pas sélectionner un modèle sur la réserve** et ensuite déclarer la même réserve preuve indépendante de la sélection. La valeur des résultats est comparative et exploratoire.

In [1]:
from pathlib import Path
import json, hashlib, os
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicHermiteSpline
from scipy.optimize import least_squares
import matplotlib.pyplot as plt

NAME = 'GVH_CHH_Jupiter_Horizons_H0_Force_Ablation_And_Time_Window_Robustness_Gate_FAST'
ROOT = Path('/content') if Path('/content').exists() else Path('/mnt/data')
SOURCE = ROOT/'GVH_CHH_JUPITER_FULL_FORCE'
OUT = ROOT/'GVH_CHH_JUPITER_H0_ABLATION'
OUT.mkdir(exist_ok=True, parents=True)
PARENT = 'GVH_CHH_Jupiter_Horizons_Full_Force_Model_And_Residual_Calibration_Gate_FAST'
MOONS = ('Io','Europa','Ganymede','Callisto')
TARGETS = ('Io','Ganymede')
ID = {'Io':501,'Europa':502,'Ganymede':503,'Callisto':504,'Sun':10}
TRAIN_DAYS = (10.0, 15.0)
EXTRA = os.environ.get('CHH_EXTENDED_ABLATION','0') == '1'
if EXTRA: TRAIN_DAYS = (10.0,12.0,15.0)
MODELS = ('KEPLER','J2_ONLY','MUTUAL_ONLY','J2_MUTUAL',
          'J2_MUTUAL_SUN','J2_J4_J6_MUTUAL_SUN')
if EXTRA: MODELS += ('J2_J4_J6_MUTUAL',)
COMMON_RESERVED_START_D = 18.0
MAX_NFEV = int(os.environ.get('CHH_ABLATION_MAX_NFEV','14'))
GM_J=126686534.0
R_J=71492.0
GM_MOONS=np.array([5959.916,3202.739,9887.834,7179.29],float)
GM_SUN=132712440041.93938
J2=0.0146965735;J4=-0.0005866085;J6=0.0000342007
RA,DEC=np.deg2rad([268.056595,64.495303])
POLE=np.array([np.cos(DEC)*np.cos(RA),np.cos(DEC)*np.sin(RA),np.sin(DEC)])
print('Source:',SOURCE,'Exists:',SOURCE.exists())
print('Training windows (days):',TRAIN_DAYS,'Models:',MODELS)

Source: /content/GVH_CHH_JUPITER_FULL_FORCE Exists: False
Training windows (days): (10.0, 15.0) Models: ('KEPLER', 'J2_ONLY', 'MUTUAL_ONLY', 'J2_MUTUAL', 'J2_MUTUAL_SUN', 'J2_J4_J6_MUTUAL_SUN')


### Import des sources depuis Colab

Le notebook précédent a écrit ses fichiers dans `/content/GVH_CHH_JUPITER_FULL_FORCE/`. Il faut **conserver le dossier** ou déposer un ZIP contenant ces fichiers dans `/content/`. Si le dossier n'existe plus, relancer le prédécesseur en Colab avec accès au réseau, puis sauvegarder son dossier. La cellule suivante s'arrête explicitement si une source manque. Elle n'effectue **aucune requête réseau** et ne produit **aucune pseudo-observation**.

In [3]:

from google.colab import drive
from pathlib import Path
import zipfile

# Connecter Google Drive à Colab
drive.mount('/content/drive')

# Rechercher notre archive scientifique
nom = "GVH_CHH_JUPITER_FULL_FORCE_RESULTS.zip"
racine = Path("/content/drive/MyDrive")

archives = list(racine.rglob(nom))

if not archives:
    raise FileNotFoundError(
        "ZIP introuvable dans Mon Drive. "
        "Vérifiez le compte Google connecté."
    )

archive = archives[0]
print("Archive trouvée :", archive)

# Extraire les données dans /content
with zipfile.ZipFile(archive) as z:
    for fichier in z.namelist():
        destination = (Path("/content") / fichier).resolve()
        if not destination.is_relative_to(
            Path("/content").resolve()
        ):
            raise ValueError("Chemin ZIP non sécurisé")
    z.extractall("/content")

# Vérifier les fichiers nécessaires
dossier = Path("/content/GVH_CHH_JUPITER_FULL_FORCE")
lunes = ["Io", "Europa", "Ganymede", "Callisto", "Sun"]

for lune in lunes:
    fichier = dossier / f"HORIZONS_{lune}_ICRF.csv"
    assert fichier.exists(), f"Manquant : {fichier.name}"
    print("OK :", fichier.name)

rapport = dossier / (
    "GVH_CHH_Jupiter_Horizons_Full_Force_Model_And_"
    "Residual_Calibration_Gate_FAST_results.json"
)
assert rapport.exists(), "Rapport JSON manquant"

print("SUCCÈS : 5 CSV et rapport JSON disponibles.")


Mounted at /content/drive
Archive trouvée : /content/drive/MyDrive/Saved from Chrome/GVH_CHH_JUPITER_FULL_FORCE_RESULTS.zip
OK : HORIZONS_Io_ICRF.csv
OK : HORIZONS_Europa_ICRF.csv
OK : HORIZONS_Ganymede_ICRF.csv
OK : HORIZONS_Callisto_ICRF.csv
OK : HORIZONS_Sun_ICRF.csv
SUCCÈS : 5 CSV et rapport JSON disponibles.


In [4]:
import zipfile
archive = ROOT/'GVH_CHH_JUPITER_FULL_FORCE_RESULTS.zip'
if not SOURCE.exists() and archive.exists():
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            # prevent path traversal before extraction
            target=(ROOT/entry.filename).resolve()
            if not target.is_relative_to(ROOT.resolve()):
                raise RuntimeError('Unsafe ZIP path: '+entry.filename)
        z.extractall(ROOT)
expected=[SOURCE/f'HORIZONS_{name}_ICRF.csv' for name in ID]
parent_json=SOURCE/(PARENT+'_results.json')
missing=[str(p) for p in expected+[parent_json] if not p.exists()]
if missing:
    raise FileNotFoundError('Missing real JPL inputs. First export the predecessor Colab directory: '+str(missing))

def sha256(p):return hashlib.sha256(p.read_bytes()).hexdigest()
PARENT_REPORT=json.loads(parent_json.read_text())
if PARENT_REPORT.get('status')!='JPL_EPHEMERIS_DIAGNOSTIC_DONE_NO_HIDDEN_BODIES':
    raise RuntimeError('Unexpected predecessor evidence status: '+str(PARENT_REPORT.get('status')))
if not PARENT_REPORT.get('all_fits_converged') or PARENT_REPORT.get('hidden_bodies_allowed') is not False:
    raise RuntimeError('Parent report did not certify required H0 calibration')
inputs={}
for key in ID:
    p=SOURCE/f'HORIZONS_{key}_ICRF.csv'
    df=pd.read_csv(p)
    need=['JD_TDB','x_km','y_km','z_km','vx_km_s','vy_km_s','vz_km_s']
    if list(df.columns)!=need:raise ValueError('Unexpected Horizons columns: '+key)
    arr=df.to_numpy(dtype=float)
    if arr.ndim!=2 or arr.shape[1]!=7 or arr.shape[0]!=97 or not np.all(np.isfinite(arr)):
        raise ValueError('Invalid JPL CSV: '+key+' '+str(arr.shape))
    inputs[key]=arr
jd=inputs['Io'][:,0]
for key,arr in inputs.items():
    if not np.allclose(arr[:,0],jd,atol=1e-8,rtol=0):
        raise ValueError('Nonmatching time grid: '+key)
if not np.allclose(np.diff(jd),0.25,atol=2e-7,rtol=0):raise ValueError('Expected 6h cadence')
source_hashes={p.name:sha256(p) for p in expected+[parent_json]}
epochs=(jd-jd[0])*86400.0
obs=np.stack([inputs[key][:,1:4] for key in MOONS],axis=1)
x0=np.stack([inputs[key][0,1:] for key in MOONS],axis=0)
sun=inputs['Sun']; sun_spline=CubicHermiteSpline(epochs,sun[:,1:4],sun[:,4:7],axis=0)
assert max(TRAIN_DAYS)<COMMON_RESERVED_START_D < epochs[-1]/86400
assert all(len(np.flatnonzero(epochs/86400 <= end))>=30 for end in TRAIN_DAYS)
print('REAL JPL EPHEMERIS FILES VERIFIED:',len(inputs),'samples each:',len(jd))
print('Provenance hashes:',source_hashes)

REAL JPL EPHEMERIS FILES VERIFIED: 5 samples each: 97
Provenance hashes: {'HORIZONS_Io_ICRF.csv': '676823feae0356ff88f601b07e5112e7146c64a55443bb1f89084d3746c9d12b', 'HORIZONS_Europa_ICRF.csv': '93a37a817d08cb3d73beeb3e9e85a5c11027e5de76e9ee40be6aa89da1f34624', 'HORIZONS_Ganymede_ICRF.csv': '51ac0ac9a11e3e96748408f9f8e5e777ef61a3af100faa8e7f7889ae28b729a4', 'HORIZONS_Callisto_ICRF.csv': 'ddd706c952bd8f58a25bd631dbf4ab8b5d71532d66cb867a8763e113d4e726b0', 'HORIZONS_Sun_ICRF.csv': 'a72c6aa94a96e8142bade51dd02bce14c874d7bf5883b89063095601478f9557', 'GVH_CHH_Jupiter_Horizons_Full_Force_Model_And_Residual_Calibration_Gate_FAST_results.json': '98ae31032c9edf16c394c26b6094545ef8dbb52fbbd540e18c73f001b6533ed9'}


## Niveau 2 — Accélérations physiques et tests de cohérence

L'accélération comprend le monopôle jovien plus, suivant l'ablation, les harmoniques zonales J2/J4/J6, les forces newtoniennes mutuelles entre les quatre lunes et la force de marée solaire prescrite. Cette modélisation n'inclut notamment pas les marées complètes, tous les satellites, les perturbateurs planétaires, la rotation détaillée du pôle ni l'incertitude de l'ajustement JPL. La trajectoire solaire dépend de JPL et n'est pas déduite de CHH.

In [5]:
def legendre_even(n,s):
    if n==2: return (3*s*s-1)/2, 3*s
    if n==4: return (35*s**4-30*s*s+3)/8,(140*s**3-60*s)/8
    if n==6: return (231*s**6-315*s**4+105*s*s-5)/16,(1386*s**5-1260*s**3+210*s)/16
    raise ValueError(n)

def zonal_potential(r, orders=(2,4,6)):
    rr=np.linalg.norm(r); s=np.dot(POLE,r)/rr
    J={2:J2,4:J4,6:J6}
    return -GM_J/rr*(1-sum(J[n]*(R_J/rr)**n*legendre_even(n,s)[0] for n in orders))

def zonal_accel(r,orders=(2,4,6)):
    rr=np.linalg.norm(r); rh=r/rr; s=np.dot(POLE,rh)
    J={2:J2,4:J4,6:J6}; a=np.zeros(3)
    for n in orders:
        P,dP=legendre_even(n,s)
        a+=GM_J*J[n]*R_J**n/rr**(n+2)*((n+1)*P*rh-dP*(POLE-s*rh))
    return a

def grad_check():
    x=np.array([6.2e5,-3e5,4.4e4]); h=8.0; eye=np.eye(3)
    grad=np.array([(zonal_potential(x+h*eye[k])-zonal_potential(x-h*eye[k]))/(2*h) for k in range(3)])
    expected=-GM_J*x/np.linalg.norm(x)**3+zonal_accel(x)
    return np.linalg.norm(-grad-expected)/np.linalg.norm(expected)
assert grad_check() < 3e-6

ABLATION={
 'KEPLER':((),False,False),
 'J2_ONLY':((2,),False,False),
 'MUTUAL_ONLY':((),True,False),
 'J2_MUTUAL':((2,),True,False),
 'J2_MUTUAL_SUN':((2,),True,True),
 'J2_J4_J6_MUTUAL':((2,4,6),True,False),
 'J2_J4_J6_MUTUAL_SUN':((2,4,6),True,True)
}

def rhs(t,y,variant):
    orders,mutual,solar=ABLATION[variant]
    st=y.reshape(4,6); pos=st[:,:3]; vel=st[:,3:]; acc=np.zeros((4,3))
    for i in range(4):
        r=pos[i];rad=np.linalg.norm(r)
        acc[i]=-GM_J*r/rad**3+zonal_accel(r,orders)
        if mutual:
            for j in range(4):
                if i==j:continue
                d=pos[j]-r
                acc[i]+=GM_MOONS[j]*(d/np.linalg.norm(d)**3-pos[j]/np.linalg.norm(pos[j])**3)
        if solar:
            rs=sun_spline(t); d=rs-r
            acc[i]+=GM_SUN*(d/np.linalg.norm(d)**3-rs/np.linalg.norm(rs)**3)
    return np.column_stack((vel,acc)).reshape(-1)

def propagate(states,tt,variant):
    tt=np.asarray(tt,float)
    if abs(tt[0])>1e-8 or not np.all(np.diff(tt)>0):raise ValueError('invalid times')
    sol=solve_ivp(lambda t,y:rhs(t,y,variant),(0,float(tt[-1])),states.ravel(),
                  method='DOP853',t_eval=tt,rtol=3e-11,atol=4e-10,max_step=21600.)
    if not sol.success:raise RuntimeError(sol.message)
    return sol.y.T.reshape(len(tt),4,6)

# Offline integrator smoke: circular orbit remains approximately circular under Kepler.
synthetic=np.zeros((4,6))
for k,r in enumerate((420000.,670000.,1070000.,1880000.)):
    synthetic[k,:3]=[r,0,0];synthetic[k,3:]=[0,np.sqrt(GM_J/r),0]
test=propagate(synthetic,np.linspace(0,2*86400,9),'KEPLER')
radial_error=np.max(np.abs(np.linalg.norm(test[:,0,:3],axis=1)-420000.))
assert radial_error<1e-3
print('PASS force/integrator unit checks;',grad_check(),radial_error)

PASS force/integrator unit checks; 8.339531982418163e-11 2.7468428015708923e-05


## Niveau 3 — Calibrations H0, fenêtres, résidus et conservation des preuves

**Paramètres nuisibles :** seules les 3 positions et 3 vitesses initiales d'Io et Ganymède sont ajustées, soit 12 paramètres par essai. Les autres états initiaux et coefficients physiques restent fixés. Aucune information issue de la réserve ne rentre dans la minimisation.  
**Sorties :** tableau d'ablation et des fenêtres CSV, rapport JSON, graphes, SHA-256 des fichiers sources. L'optimiseur doit rapporter explicitement une convergence. Toute réserve contient des éphémérides JPL, non des mesures brutes indépendantes.

In [6]:
days=epochs/86400.0
COMMON=np.flatnonzero(days>COMMON_RESERVED_START_D)
TARGET_IDX=[0,2]
REPORT={
  'notebook':NAME,'state':'JPL_H0_ABLATION_RUNNING','parent':PARENT,
  'parent_json_sha256':sha256(parent_json), 'source_hashes_sha256':source_hashes,
  'h0_hidden_body_count':0,'GVH_USED':False,'EOM_GVH':False,'ALMA_USED':False,
  'target_moons':list(TARGETS),'train_windows_days':list(TRAIN_DAYS),
  'common_reserved_start_day_exclusive':COMMON_RESERVED_START_D,
  'model_names':list(MODELS),'nuisance_parameter_count':12,
  'external_sun_forcing_uses_reserved_ephemerides':True,
  'holdout_is_JPL_reference_ephemeris_not_independent_raw_astrometry':True,
  'results':[], 'nonlinear_optimizer_nfev_budget':MAX_NFEV
}

def initial_shift(z):
    v=x0.copy()
    shift=np.asarray(z).reshape(2,6)
    for j,i in enumerate(TARGET_IDX):
        v[i,:3]+=1000*shift[j,:3]
        v[i,3:]+=0.01*shift[j,3:]
    return v

def fit_resid(z,ix,variant):
    trial=propagate(initial_shift(z),epochs[ix],variant)
    return ((trial[:,TARGET_IDX,:3]-obs[ix][:,TARGET_IDX,:])/1000).reshape(-1)

full_rows=[]; curves={}
for cutoff in TRAIN_DAYS:
    train=np.flatnonzero(days<=cutoff+1e-9)
    assert max(train)<min(COMMON)
    nearby=np.flatnonzero((days>cutoff)&(days<=cutoff+3.0))
    print('\nTraining cutoff:',cutoff,'day | n_train',len(train),'common reserved',len(COMMON))
    for variant in MODELS:
        fit=least_squares(fit_resid,np.zeros(12),args=(train,variant),
                          method='trf',max_nfev=MAX_NFEV,
                          xtol=1e-6,ftol=1e-6,gtol=1e-6)
        trial=propagate(initial_shift(fit.x),epochs,variant)
        res=trial[:,TARGET_IDX,:3]-obs[:,TARGET_IDX,:]
        rmse=lambda inds,v: float(np.sqrt(np.mean(v[inds]**2)))
        one={k:rmse(COMMON,res[:,j,:]) for j,k in enumerate(TARGETS)}
        entry={
          'window_train_days':float(cutoff),'model':variant,
          'optimizer_success':bool(fit.success),'optimizer_message':str(fit.message),
          'optimizer_nfev':int(fit.nfev),
          'train_RMSE_xyz_km':rmse(train,res),
          'common_reserved_RMSE_xyz_km':rmse(COMMON,res),
          'nearby_3d_RMSE_xyz_km':rmse(nearby,res) if len(nearby) else None,
          'Io_reserved_RMSE_xyz_km':one['Io'],
          'Ganymede_reserved_RMSE_xyz_km':one['Ganymede'],
          'initial_state_corrections_scaled':fit.x.tolist(),
          'training_samples':int(len(train)),
          'reserved_samples':int(len(COMMON)),
        }
        REPORT['results'].append(entry)
        full_rows.append({k:v for k,v in entry.items() if k!='initial_state_corrections_scaled'})
        curves[(float(cutoff),variant)]=np.sqrt(np.mean(res**2,axis=(1,2)))
        print(variant, 'reserved RMSE km',round(entry['common_reserved_RMSE_xyz_km'],4),
              'Io',round(one['Io'],3),'Ganymede',round(one['Ganymede'],3),'fit_ok',bool(fit.success))

df=pd.DataFrame(full_rows)
df.to_csv(OUT/(NAME+'_metrics.csv'),index=False)
REPORT['all_optimizer_runs_converged']=bool(all(x['optimizer_success'] for x in REPORT['results']))
REPORT['state']='JPL_H0_ABLATION_EXECUTED' if REPORT['all_optimizer_runs_converged'] else 'INCOMPLETE_NONCONVERGED_OPTIMIZERS'
print(df[['window_train_days','model','train_RMSE_xyz_km','common_reserved_RMSE_xyz_km','Io_reserved_RMSE_xyz_km','Ganymede_reserved_RMSE_xyz_km','optimizer_success']].to_string(index=False))


Training cutoff: 10.0 day | n_train 41 common reserved 24
KEPLER reserved RMSE km 279.7615 Io 330.051 Ganymede 218.173 fit_ok True
J2_ONLY reserved RMSE km 312.0178 Io 384.931 Ganymede 215.727 fit_ok True
MUTUAL_ONLY reserved RMSE km 73.4725 Io 83.341 Ganymede 62.054 fit_ok True
J2_MUTUAL reserved RMSE km 19.2011 Io 4.255 Ganymede 26.819 fit_ok True
J2_MUTUAL_SUN reserved RMSE km 18.392 Io 4.27 Ganymede 25.657 fit_ok True
J2_J4_J6_MUTUAL_SUN reserved RMSE km 18.3752 Io 4.186 Ganymede 25.647 fit_ok True

Training cutoff: 15.0 day | n_train 61 common reserved 24
KEPLER reserved RMSE km 244.5754 Io 278.162 Ganymede 205.573 fit_ok True
J2_ONLY reserved RMSE km 266.9034 Io 323.644 Ganymede 194.241 fit_ok True
MUTUAL_ONLY reserved RMSE km 61.2673 Io 75.784 Ganymede 42.002 fit_ok True
J2_MUTUAL reserved RMSE km 13.1701 Io 4.029 Ganymede 18.184 fit_ok True
J2_MUTUAL_SUN reserved RMSE km 13.1763 Io 4.038 Ganymede 18.191 fit_ok True
J2_J4_J6_MUTUAL_SUN reserved RMSE km 13.16 Io 3.959 Ganymede 1

### Comparaisons conditionnelles d'ablation

Les différences d'erreur peuvent être positives ou négatives. Elles ne donnent pas une décomposition additive des effets physiques : les paramètres sont réoptimisés et les forces peuvent interagir. Une réduction des résidus sur éphémérides ne démontre pas une découverte gravitationnelle. Nous ne choisissons pas une nouvelle « meilleure » formule à partir de la réserve sans un **autre** jeu de test indépendant.

In [7]:
# Quantify conditional ablation at each training window, without retroactive selection.
contrast_pairs=(
 ('J2 contribution without mutual','KEPLER','J2_ONLY'),
 ('Mutual contribution without J2','KEPLER','MUTUAL_ONLY'),
 ('Mutual contribution with J2','J2_ONLY','J2_MUTUAL'),
 ('J2 contribution with mutual','MUTUAL_ONLY','J2_MUTUAL'),
 ('Sun contribution given J2+mutual','J2_MUTUAL','J2_MUTUAL_SUN'),
 ('J4/J6 contribution given J2+mutual+Sun','J2_MUTUAL_SUN','J2_J4_J6_MUTUAL_SUN'),
)
rows=[]
for window in TRAIN_DAYS:
    selected={x['model']:x for x in REPORT['results'] if x['window_train_days']==window}
    for name,base,enh in contrast_pairs:
        r0=selected[base]['common_reserved_RMSE_xyz_km'];r1=selected[enh]['common_reserved_RMSE_xyz_km']
        rows.append(dict(train_days=window,contrast=name,base=base,enhanced=enh,
                         base_rmse_km=r0,enhanced_rmse_km=r1,
                         reduction_percent=100*(r0-r1)/r0))
contrast=pd.DataFrame(rows)
contrast.to_csv(OUT/(NAME+'_contrasts.csv'),index=False)
print(contrast.to_string(index=False))


print('Contrastes CSV enregistrés ; graphes reportés après le verdict technique.')


 train_days                               contrast          base            enhanced  base_rmse_km  enhanced_rmse_km  reduction_percent
       10.0         J2 contribution without mutual        KEPLER             J2_ONLY    279.761463        312.017766         -11.529931
       10.0         Mutual contribution without J2        KEPLER         MUTUAL_ONLY    279.761463         73.472518          73.737442
       10.0            Mutual contribution with J2       J2_ONLY           J2_MUTUAL    312.017766         19.201090          93.846155
       10.0            J2 contribution with mutual   MUTUAL_ONLY           J2_MUTUAL     73.472518         19.201090          73.866297
       10.0       Sun contribution given J2+mutual     J2_MUTUAL       J2_MUTUAL_SUN     19.201090         18.392036           4.213580
       10.0 J4/J6 contribution given J2+mutual+Sun J2_MUTUAL_SUN J2_J4_J6_MUTUAL_SUN     18.392036         18.375196           0.091566
       15.0         J2 contribution without mutu

## Niveau 4 — Contrôles et verdict limité

**PASS technique** exige : données JPL présentes et alignées, provenance, 12 paramètres par variante, aucune lune cachée, fenêtres réservées correctement séparées, gradients/integrateur contrôlés et optimisation convergente dans toutes les variantes. Ce PASS n'est pas une validation de CHH ni de GVH. Les écarts d'ablation restent descriptifs. En cas de non-convergence, conserver le rapport et marquer le verrou ouvert.

In [8]:
required=[SOURCE/f'HORIZONS_{k}_ICRF.csv' for k in ID]
checks={
 'five_real_jpl_csv_exist': all(p.exists() for p in required),
 'all_hashes_present': all(source_hashes.get(p.name) for p in required+[parent_json]),
 'matching_time_grid': all(np.allclose(inputs[k][:,0],jd,atol=1e-8,rtol=0) for k in ID),
 'reserved_after_training': all(max(np.flatnonzero(days<=w))<min(COMMON) for w in TRAIN_DAYS),
 'fixed_12_nuisance_states': all(len(e['initial_state_corrections_scaled'])==12 for e in REPORT['results']),
 'no_hidden_bodies': REPORT['h0_hidden_body_count']==0,
 'no_gvh_or_alma': not REPORT['GVH_USED'] and not REPORT['EOM_GVH'] and not REPORT['ALMA_USED'],
 'zonal_gradient_math': bool(grad_check()<3e-6),
 'kepler_propagation_smoke': bool(radial_error<1e-3),
 'fit_convergence': REPORT['all_optimizer_runs_converged'],
 'expected_rows': len(REPORT['results'])==len(MODELS)*len(TRAIN_DAYS)
}
REPORT['checks']=checks
REPORT['scientific_verdict']=(
    'PASS_TECHNICAL_H0_FORCE_ABLATION__JPL_REFERENCE_ONLY__CHH_ORIGINALITY_OPEN'
    if all(checks.values()) else 'FAIL_OR_INCOMPLETE_H0_GATE__NO_BODY_DISCOVERY'
)
REPORT['contrasts']=contrast.to_dict(orient='records')
REPORT['residuals_are_not_observation_noise']=True
result=OUT/(NAME+'_results.json')
result.write_text(json.dumps(REPORT,ensure_ascii=False,indent=2,default=float))
(OUT/'SHA256_INPUTS.txt').write_text(''.join(f'{value}  {key}\n' for key,value in sorted(source_hashes.items())))
print('Checks:',sum(checks.values()),'/',len(checks))
print('Verdict:',REPORT['scientific_verdict'])
print('Saved:',result)
if not all(checks.values()): print('Unpassed checks:',[k for k,v in checks.items() if not v])

Checks: 11 / 11
Verdict: PASS_TECHNICAL_H0_FORCE_ABLATION__JPL_REFERENCE_ONLY__CHH_ORIGINALITY_OPEN
Saved: /content/GVH_CHH_JUPITER_H0_ABLATION/GVH_CHH_Jupiter_Horizons_H0_Force_Ablation_And_Time_Window_Robustness_Gate_FAST_results.json


### Protocole de finalisation v0.2 (sans modifier le plan scientifique)

1. Déposez `GVH_CHH_JUPITER_FULL_FORCE_RESULTS.zip` dans `/content` (ou conservez le dossier `/content/GVH_CHH_JUPITER_FULL_FORCE`).
2. **Runtime > Run all** : les douze ajustements sont recalculés. Les graphiques ne peuvent plus bloquer la cellule de verdict.
3. Vérifiez la sortie de l'audit : `Checks: 11 / 11`, le verdict limité et l'existence du JSON et des CSV.
4. Exécutez les cellules annexes facultatives pour graphique et ZIP, puis téléchargez `GVH_CHH_JUPITER_H0_ABLATION_FINAL_RESULTS.zip` via `from google.colab import files; files.download(archive_name)`.
5. Toute erreur ou absence de convergence reste un échec technique / audit incomplet : ne pas publier de PASS.

Les valeurs sont des comparaisons à des éphémérides JPL calculées ; elles ne valident ni une lune nouvelle ni une loi GVH.


In [9]:
# Graphique facultatif APRES emission du verdict et du JSON, désactivé par défaut.
# Pour activer : os.environ['CHH_EXPORT_PLOT']='1' avant cette cellule.
if os.environ.get('CHH_EXPORT_PLOT','0') == '1':
    try:
        import matplotlib
        matplotlib.use('Agg', force=True)
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(8,4))
        for cutoff in TRAIN_DAYS:
            sub=df[df.window_train_days==cutoff]
            ax.plot(sub.model, sub.common_reserved_RMSE_xyz_km, marker='o', label=f'Entrainement {cutoff:g} j')
        ax.set_yscale('log')
        ax.set_ylabel('RMSE sur reserve >18 j (km)')
        ax.tick_params(axis='x', labelrotation=55)
        ax.legend(fontsize=8)
        fig.tight_layout()
        fig.savefig(OUT/(NAME+'_diagnostics.png'),dpi=95,bbox_inches='tight')
        plt.close(fig)
        print('Graphique PNG enregistre')
    except Exception as e:
        print('GRAPHIQUE NON DISPONIBLE, VERDICT NUMERIQUE CONSERVE :', repr(e))
else:
    print('Graphique facultatif non execute : le rapport JSON et les CSV constituent les preuves numeriques.')


Graphique facultatif non execute : le rapport JSON et les CSV constituent les preuves numeriques.


In [10]:
# Constituer le ZIP de preuves APRES le verdict. Aucun nouveau calcul scientifique ici.
import shutil
assert 'scientific_verdict' in REPORT and result.exists(), 'Rapport final non cree : executer toutes les cellules dans l ordre.'
archive_name = shutil.make_archive(str(ROOT/'GVH_CHH_JUPITER_H0_ABLATION_FINAL_RESULTS'),'zip',root_dir=str(ROOT),base_dir=OUT.name)
print('ZIP final :', archive_name)
print('Statut canonique :', REPORT['scientific_verdict'])
print('11 controles :', sum(REPORT['checks'].values()), '/', len(REPORT['checks']))
# Colab : activer si souhaite : from google.colab import files; files.download(archive_name)


ZIP final : /content/GVH_CHH_JUPITER_H0_ABLATION_FINAL_RESULTS.zip
Statut canonique : PASS_TECHNICAL_H0_FORCE_ABLATION__JPL_REFERENCE_ONLY__CHH_ORIGINALITY_OPEN
11 controles : 11 / 11


In [11]:

from pathlib import Path
from google.colab import drive
import shutil
import hashlib

source = Path(
    "/content/GVH_CHH_JUPITER_H0_ABLATION_FINAL_RESULTS.zip"
)
assert source.is_file(), "ZIP final introuvable."

drive.mount("/content/drive", force_remount=False)

dossier = Path("/content/drive/MyDrive/GVH_CHH_Jupiter_Archives")
dossier.mkdir(parents=True, exist_ok=True)

destination = dossier / source.name
shutil.copy2(source, destination)

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

assert sha256(source) == sha256(destination)
print("Archive enregistrée et intégrité vérifiée :", destination)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive enregistrée et intégrité vérifiée : /content/drive/MyDrive/GVH_CHH_Jupiter_Archives/GVH_CHH_JUPITER_H0_ABLATION_FINAL_RESULTS.zip
